# Filter From Scratch!

At this point, you know much more about GNC than I did during my first semester on the GNC team. It's time to take off the training wheels and **build your own filter from scratch.**

Your filter should have, at minimum:

**Initialization step** — set your initial state, coefficients, covariance, etc.
**Predict step** — predict the next state.
**Update step** — use the measurement to update your prediction.

Finally, you'll need to **test your filter.**

### Goal

For the provided dataset, your filter should achieve an RMS error below X.

# Important Note: 

**Your measurment vector $z$ is defined as:** 

$z = \begin{bmatrix} t_{seconds} \\ a_{x-m/s} \\ a_{y-m/s} \\ a_{z-m/s} \end{bmatrix}$ 

In [ ]:
# Do not change this! 
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
import copy 
import os


# Initialize Step: 

In [ ]:
#% exercise
#% title: Initialize Step
#% checker: initialize_step
### BEGIN STUB
class filter_scratch: 
    def __init__(self): 
        # modify the header to include the arguments passed into your filter. i.e def __init__(self, alpha, beta, gamma): etc
        # set all arguments into your filter as attributes of the class, using the format self.argument_name = argument_value 
        # make sure each argument has self: self.alpha = 0.1 is the format 
        self.state = np.zeros(6)   # for now lets assume this is np.array([position_x, velocity_x, position_y, velocity_y, position_z, velocity_z])
### END STUB
### BEGIN SOLUTION
class filter_scratch:
    """One reference answer: an alpha-beta filter on the measured acceleration.

    It is *a* correct answer, not *the* correct answer — a complementary filter
    or a full Kalman filter would be graded just as happily. All the checkers
    ask is that the tuning knobs live on the filter and that `state` is the six
    numbers the SILSIM cell plots.
    """

    def __init__(self, alpha=0.25, beta=0.15, dt=0.095, gravity=9.80665):
        self.alpha = alpha          # how hard a residual corrects the velocity
        self.beta = beta            # how fast the acceleration estimate follows the sensor
        self.dt = dt                # fallback step, for the very first sample
        self.bias = np.array([gravity, 0.0, 0.0])  # on the pad the sensor reads +1 g on x
        self.accel = np.zeros(3)    # current acceleration estimate, m/s^2
        self.last_t = None          # timestamp of the previous sample
        self.last_dt = dt           # the step predict() just took, reused by update()
        self.state = np.zeros(6)    # [position_x, velocity_x, position_y, velocity_y, position_z, velocity_z]
### END SOLUTION

# Predict Step: 

In [ ]:
#% exercise
#% title: Predict Step
#% checker: predict_step
### BEGIN STUB
def predict(self, measurement):
    self.state = np.zeros(6) 
    # pull your variables to be used in the predict function from the class attributes using self.variable_name
    # compute a prediction of the state based on the previous state and the current input, using the attributes of the class as needed
### END STUB
### BEGIN SOLUTION
def predict(self, measurement):
    # How long since the previous sample? The timestamp is measurement[0].
    t = float(measurement[0])
    dt = self.dt if self.last_t is None else t - self.last_t
    if not 0.0 < dt < 10.0:      # first sample, or a gap in the log
        dt = self.dt
    self.last_t, self.last_dt = t, dt

    # Constant-acceleration propagation, one axis triple at a time:
    # positions live at 0, 2, 4 and velocities at 1, 3, 5.
    position = self.state[0::2] + self.state[1::2] * dt + 0.5 * self.accel * dt ** 2
    velocity = self.state[1::2] + self.accel * dt

    predicted = np.zeros(6)
    predicted[0::2] = position
    predicted[1::2] = velocity
    self.state = predicted
### END SOLUTION


# DO NOT TOUCH THIS LINE. It is used to bind the predict function to the filter_scratch class.
filter_scratch.predict = predict

# Update Step: 

In [ ]:
#% exercise
#% title: Update Step
#% checker: update_step
### BEGIN STUB
def update(self, measurement):
    # pull your variables to be used in the update function from the class attributes using self.variable_name
    # compute an update of the state based on the previous state and the current input, using the attributes of the class as needed
    # finally return the state  
    self.state = np.zeros(6)
### END STUB
### BEGIN SOLUTION
def update(self, measurement):
    # The measurement is [t, ax, ay, az]; take the 1 g the sensor reads at rest
    # back out before believing the accelerations.
    measured = np.asarray(measurement[1:4], dtype=float) - self.bias

    # The part of the reading predict() did not already account for.
    residual = measured - self.accel

    # beta blends it into the acceleration estimate the next predict will use,
    # alpha lets it correct the velocity we just propagated.
    self.accel = self.accel + self.beta * residual
    self.state[1::2] = self.state[1::2] + self.alpha * residual * self.last_dt
### END SOLUTION

# DO NOT TOUCH THIS LINE. It is used to bind the update function to the filter_scratch class.
filter_scratch.update = update

# Once you're ready to test...run this code and see the output! 

In [ ]:
# This is just a SILSIM to test your filter. Do not modify this function. It will be used to test your filter.

filter = filter_scratch() # Make sure to modify this line to include the arguments needed to initialize your filter. i.e filter = filter_scratch(alpha, beta, gamma) etc
csv_path = './static/SAWA_Decimate.csv' 
data = pd.read_csv(csv_path).to_numpy()
time_plot = data[:, 0]
state_plot = []

def SILSIM(filter, input_data):
    for i in range(len(input_data)):
        filter.predict(input_data[i])
        filter.update(input_data[i])
        state_plot.append(copy.deepcopy(filter.state))

# Run simulation and convert recorded states to numpy array
SILSIM(filter, data)
state_plot = np.array(state_plot)

# Plotting: Position, Velocity, and Measured Acceleration side-by-side
fig, axes = plt.subplots(3, 3, figsize=(16, 10), sharex=True)

# --- X-Axis ---
axes[0, 0].plot(time_plot, state_plot[:, 0], label='Position X', color='tab:blue')
axes[0, 0].set_ylabel('Position (m)')
axes[0, 0].set_title('X-Axis Position')
axes[0, 0].grid(True)
axes[0, 0].legend()

axes[0, 1].plot(time_plot, state_plot[:, 1], label='Velocity X', color='tab:orange')
axes[0, 1].set_ylabel('Velocity (m/s)')
axes[0, 1].set_title('X-Axis Velocity')
axes[0, 1].grid(True)
axes[0, 1].legend()

axes[0, 2].plot(time_plot, data[:, 1], label='Measured Accel X', color='tab:green', alpha=0.7)
axes[0, 2].set_ylabel('Acceleration (m/s²)')
axes[0, 2].set_title('X-Axis Measured Acceleration')
axes[0, 2].grid(True)
axes[0, 2].legend()

# --- Y-Axis ---
axes[1, 0].plot(time_plot, state_plot[:, 2], label='Position Y', color='tab:blue')
axes[1, 0].set_ylabel('Position (m)')
axes[1, 0].set_title('Y-Axis Position')
axes[1, 0].grid(True)
axes[1, 0].legend()

axes[1, 1].plot(time_plot, state_plot[:, 3], label='Velocity Y', color='tab:orange')
axes[1, 1].set_ylabel('Velocity (m/s)')
axes[1, 1].set_title('Y-Axis Velocity')
axes[1, 1].grid(True)
axes[1, 1].legend()

axes[1, 2].plot(time_plot, data[:, 2], label='Measured Accel Y', color='tab:green', alpha=0.7)
axes[1, 2].set_ylabel('Acceleration (m/s²)')
axes[1, 2].set_title('Y-Axis Measured Acceleration')
axes[1, 2].grid(True)
axes[1, 2].legend()

# --- Z-Axis ---
axes[2, 0].plot(time_plot, state_plot[:, 4], label='Position Z', color='tab:blue')
axes[2, 0].set_xlabel('Time (s)')
axes[2, 0].set_ylabel('Position (m)')
axes[2, 0].set_title('Z-Axis Position')
axes[2, 0].grid(True)
axes[2, 0].legend()

axes[2, 1].plot(time_plot, state_plot[:, 5], label='Velocity Z', color='tab:orange')
axes[2, 1].set_xlabel('Time (s)')
axes[2, 1].set_ylabel('Velocity (m/s)')
axes[2, 1].set_title('Z-Axis Velocity')
axes[2, 1].grid(True)
axes[2, 1].legend()

axes[2, 2].plot(time_plot, data[:, 3], label='Measured Accel Z', color='tab:green', alpha=0.7)
axes[2, 2].set_xlabel('Time (s)')
axes[2, 2].set_ylabel('Acceleration (m/s²)')
axes[2, 2].set_title('Z-Axis Measured Acceleration')
axes[2, 2].grid(True)
axes[2, 2].legend()

plt.tight_layout()
plt.show()
